# 13장 동적 메모리 관리와 표준 함수
본 노트북은 C 언어의 **동적 메모리 관리 기법**과 C 표준 라이브러리에서 제공하는 **표준 함수(수학, 문자열 변환, 문자 평가 및 변환)**의 개념을 정리하고, 실습 예제 코드를 분석합니다.

## 13.1 동적 메모리 관리 개요

### 1. 메모리 할당 방식 비교

| 구분 | 정적 메모리 할당 (Static Allocation) | 동적 메모리 할당 (Dynamic Allocation) |
| :--- | :--- | :--- |
| **할당 시기** | 컴파일 타임 (Compile-time) | 런타임 (Run-time / 프로그램 실행 중) |
| **메모리 크기** | 프로그램 작성 시 미리 정해짐 (고정) | 실행 중 사용자의 요구에 따라 자유롭게 변함 |
| **메모리 영역** | 데이터(Data) 영역, 스택(Stack) 영역 | 힙(Heap) 영역 |
| **특징** | 메모리 크기가 부족하면 소스 수정 필요, 너무 크면 낭비 발생 | 필요할 때 할당받고 불필요할 때 반납하여 효율적임 |

### 2. C 프로그램의 메모리 4대 영역
- **프로그램(Text) 영역**: 실행할 기계어 코드와 명령어가 위치하는 영역.
- **데이터(Data) 영역**: 전역 변수, 정적(Static) 변수, 상수 등이 할당되는 영역. 프로그램 시작 시 생성되며 종료 시 소멸됩니다.
- **힙(Heap) 영역**: 프로그래머가 필요에 따라 동적으로 메모리를 할당받는 영역. 런타임에 크기가 결정됩니다.
- **스택(Stack) 영역**: 함수 호출과 관련된 지역 변수, 매개 변수, 리턴 값 등이 저장되는 영역. 함수가 종료되면 자동으로 소멸됩니다.

## 13.2 동적 메모리 관리 함수 (`stdlib.h`)

동적 메모리를 관리하기 위한 4가지 주요 함수가 존재합니다. 이 함수들은 `<stdlib.h>` 헤더 파일에 선언되어 있습니다. 본 코드들은 `day2/01_dynamic_memory/` 폴더에 분류되어 있습니다.

### 1. `malloc` (Memory Allocation)
- **원형**: `void *malloc(size_t size);`
- **설명**: `size` 바이트 크기만큼의 메모리를 힙 영역에 할당하고 시작 주소를 반환합니다. 초기화되지 않은 쓰레기 값이 들어있습니다.
- **반환값**: 정상 실행 시 할당된 영역의 시작 주소(void *), 실패 시 `NULL` 반환.

### 2. `free` (Free Memory)
- **원형**: `void free(void *ptr);`
- **설명**: `malloc`, `calloc`, `realloc` 등을 통해 동적 할당된 포인터 `ptr`이 가리키는 메모리를 해제합니다. 해제하지 않으면 메모리 누수(Memory Leak)가 발생합니다.

### 3. `calloc` (Clear Allocation)
- **원형**: `void *calloc(size_t num, size_t size);`
- **설명**: `size` 바이트 크기의 공간을 `num` 개수만큼(즉, `num * size` 바이트) 할당하고, **모든 비트를 0으로 초기화**합니다.
- **반환값**: 할당된 메모리의 시작 주소, 실패 시 `NULL` 반환.

### 4. `realloc` (Re-allocation)
- **원형**: `void *realloc(void *pointer, size_t size);`
- **설명**: 이미 동적 할당된 포인터 `pointer`가 가리키는 메모리 영역의 크기를 `size` 바이트로 재조정합니다. 기존 데이터는 유지(크기가 줄어들 경우 초과분은 잘림)되며, 다른 위치에 할당되는 경우 기존 주소의 메모리는 자동으로 해제되고 데이터가 복사됩니다.
- **반환값**: 재할당된 메모리의 시작 주소, 실패 시 `NULL` 반환 (실패 시 원본 포인터는 그대로 유지됨).

### 예제 1: `malloc`의 기본 사용 및 포인터 연산 (`01_dynamic_memory/ex13_1.c`)
이 예제는 5개의 `int`형 공간을 할당받아 배열 인덱스 형식 `p[i]`와 포인터 연산 형식 `*(p + i)`으로 접근하는 방법을 보여줍니다.

In [ ]:
// 01_dynamic_memory/ex13_1.c
#include <stdio.h>
#include <stdlib.h>

int main() {
    int i, *p;
    p = (int *)malloc(5 * sizeof(int));
    if (p == NULL) {
        printf("메모리 할당 실패\n");
        return 1;
    }

    for (i = 0; i < 5; i++) {
        p[i] = i;
        printf("p[%d]에는 %d이 들어 있습니다. \n", i, p[i]);
    }
    printf("\n");
    for (i = 0; i < 5; i++) {
        *(p + i) = i;
        printf("*(p+%d)에는 %d이 들어 있습니다. \n", i, *(p + i));
    }
    free(p);
    return 0;
}

### 예제 2: `malloc`을 활용한 동적 문자열 처리 (`01_dynamic_memory/ex13_2.c`)
이 예제는 15글자 크기의 `char`형 포인터를 동적 할당하여 문자열을 대입하고 출력합니다.
> **주의**: macOS 및 Linux 표준 환경 호환을 위해 `strcpy_s`를 표준 `strcpy`로 매핑해주는 호환 처리를 작성했습니다.

In [ ]:
// 01_dynamic_memory/ex13_2.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#ifndef _MSC_VER
#define strcpy_s(dest, size, src) strcpy(dest, src)
#endif

int main() {
    char *sptr;
    sptr = (char *)malloc(15 * sizeof(char));
    if (sptr == NULL) {
        printf("Insufficient memory \n");
        return 1;
    }
    strcpy_s(sptr, 15, "Hello! World.");
    printf("%s \n", sptr);
    free(sptr);
    return 0;
}

### 예제 3: `calloc`을 사용한 0 초기화 메모리 할당 (`01_dynamic_memory/ex13_3.c`)
`calloc`은 할당 직후 쓰레기 값이 아닌 0으로 값을 자동으로 초기화해주므로 안전합니다.

In [ ]:
// 01_dynamic_memory/ex13_3.c
#include <stdio.h>
#include <stdlib.h>

int main() {
    int i, *p;
    p = (int *)calloc(5, sizeof(int));
    if (p == NULL) {
        printf("메모리 할당 실패\n");
        return 1;
    }
    for (i = 0; i < 5; i++) 
        printf("p[%d] : %d ", i, p[i]); // 0으로 자동 세팅됨
    printf("\n\n");
    for (i = 0; i < 5; i++) {
        p[i] = i;
        printf("p[%d]에는 %d이 들어 있습니다. \n", i, p[i]);
    }
    free(p);
    return 0;
}

### 예제 4: `realloc`을 사용한 메모리 공간 크기 재지정 (`01_dynamic_memory/ex13_4.c`)
처음 5바이트로 문자열을 할당받은 뒤, `realloc`을 통해 15바이트로 공간을 확장하여 추가 문자열을 붙이는 기법을 구현합니다.

In [ ]:
// 01_dynamic_memory/ex13_4.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#ifndef _MSC_VER
#define strcpy_s(dest, size, src) strcpy(dest, src)
#define strcat_s(dest, size, src) strcat(dest, src)
#endif

int main() {
    char *str;
    str = (char *)malloc(5 * sizeof(char));
    if (str == NULL) return 1;
    strcpy_s(str, 5, "Good");
    puts(str);
    
    // 15바이트로 재할당 (안전한 임시 포인터 사용 패턴)
    char *temp = (char *)realloc(str, 15);
    if (temp == NULL) {
        free(str);
        return 1;
    }
    str = temp;
    strcat_s(str, 15, " morning");
    puts(str);
    free(str);
    return 0;
}

### 예제 5: `realloc`을 통한 이종 자료형 크기 변경 (`01_dynamic_memory/ex13_5.c`)
`int`형 포인터로 쓰이던 주소를 `long long`형 포인터(8바이트 단위) 크기만큼 재할당하는 예제입니다.

In [ ]:
// 01_dynamic_memory/ex13_5.c
#include <stdio.h>
#include <stdlib.h>

int main() {
    int i;
    int *pa;
    long long *pb;
    
    pa = (int *)calloc(5, sizeof(int));
    if (pa == NULL) return 1;
    for (i = 0; i < 5; i++) pa[i] = i;
    for (i = 0; i < 5; i++) printf("%6d ", pa[i]);
    printf("\n");

    // int 5개(20바이트) -> long long 5개(40바이트)로 확장
    pb = (long long *)realloc(pa, 5 * sizeof(long long));
    if (pb == NULL) {
        free(pa);
        return 1;
    }
    for (i = 0; i < 5; i++) pb[i] = (long long)i * 1000;
    for (i = 0; i < 5; i++) printf("%6lld ", pb[i]);
    printf("\n");
    free(pb);
    return 0;
}

### 실습 문제 13.1 & 13.2 구현 비교
- `01_dynamic_memory/px13_1.c`: `malloc` 생성 후 `realloc` 크기 조정.
- `01_dynamic_memory/px13_2.c`: `calloc` 생성 후 `realloc` 크기 조정 (자동 0 초기화 유지 확인).

In [ ]:
// 01_dynamic_memory/px13_1.c (malloc -> realloc)
#include <stdio.h>
#include <stdlib.h>

int main() {
    int i, *pt;
    pt = (int *)malloc(5 * sizeof(int));
    if (pt == NULL) return 1;
    for (i = 0; i < 5; i++) pt[i] = 10 * i;
    for (i = 0; i < 5; i++) printf("%d ", pt[i]);
    printf("\n");
    
    int *temp = (int *)realloc(pt, 10 * sizeof(int));
    if (temp == NULL) {
        free(pt);
        return 1;
    }
    pt = temp;
    for (i = 5; i < 10; i++) pt[i] = 100 * i;
    for (i = 0; i < 10; i++) printf("%d ", pt[i]);
    printf("\n");
    free(pt);
    return 0;
}

In [ ]:
// 01_dynamic_memory/px13_2.c (calloc -> realloc)
#include <stdio.h>
#include <stdlib.h>

int main() {
    int i, *pt;
    pt = (int *)calloc(5, sizeof(int));
    if (pt == NULL) return 1;
    
    int *temp = (int *)realloc(pt, 10 * sizeof(int));
    if (temp == NULL) {
        free(pt);
        return 1;
    }
    pt = temp;
    for (i = 5; i < 10; i++) pt[i] = 100 * i;
    for (i = 0; i < 10; i++) printf("%d ", pt[i]); // 앞 5개는 0으로 채워짐
    printf("\n");
    free(pt);
    return 0;
}

## 13.3 수학연산 관련 함수 (`math.h`)
수학 연산과 관련된 함수들은 헤더 파일 `<math.h>`에 포함되어 있으며, 관련 소스들은 `day2/02_math/` 폴더에 위치해 있습니다. 대부분의 함수들이 `double`형 실수를 인수로 받고 반환합니다.

### 주요 수학 함수 정리
- `sin(x)`, `cos(x)`, `tan(x)`: 삼각함수. 인자 `x`는 라디안 단위여야 합니다. (Radian = Degree * &pi; / 180)
- `exp(x)`: 자연 상수 $e$의 지수 값 ($e^x$)을 구함.
- `log(x)`: 자연로그 $\ln x$를 구함.
- `log10(x)`: 상용로그 $\log_{10} x$를 구함.
- `sqrt(x)`: 제곱근 $\sqrt{x}$를 구함.
- `pow(x, y)`: 거듭제곱 $x^y$를 구함.
- `ceil(x)`: 올림값 (소수점 이하 올림).
- `floor(x)`: 내림값 (소수점 이하 버림).
- `fmod(x, y)`: 실수 $x$를 $y$로 나눈 나머지.
- `modf(x, *intptr)`: 실수 $x$를 소수부(반환값)와 정수부(`*intptr`)로 분할하여 분리함.
- `abs(x)`: **정수**의 절대값 구함 (단, 이 함수는 `<stdlib.h>`에 정의되어 있음).

### 예제: 삼각함수 및 기타 수학연산 예제 모음 (`02_math/ex13_6.c` ~ `02_math/ex13_9.c`, `02_math/px13_3.c`)
수학 함수 라이브러리를 활용한 예제들의 대표적인 구현체입니다.

In [ ]:
// 02_math/ex13_6.c (Trigonometric functions)
#include <stdio.h>
#include <math.h>

const double pi = 3.14159;

int main() {
    int deg1 = 30, deg2 = 45;
    double rad1, rad2;
    rad1 = deg1 * pi / 180.0;
    rad2 = deg2 * pi / 180.0;
    
    printf("sin(%d) = %lf \n", deg1, sin(rad1));
    printf("cos(%d) = %lf \n", deg1, cos(rad1));
    printf("tan(%d) = %lf \n", deg1, tan(rad1));
    printf("\n");
    printf("sin(%d) = %lf \n", deg2, sin(rad2));
    printf("cos(%d) = %lf \n", deg2, cos(rad2));
    printf("tan(%d) = %lf \n", deg2, tan(rad2));
    return 0;
}

In [ ]:
// 02_math/ex13_9.c (ceil, floor, fmod, modf)
#include <stdio.h>
#include <math.h>

int main() {
    double x1 = 65.156, x2 = 65.876;
    double y, z;
    
    printf("ceil(%lf) = %lf \n", x1, ceil(x1));
    printf("ceil(%lf) = %lf \n\n", x2, ceil(x2));
    printf("floor(%lf) = %lf \n", x1, floor(x1));
    printf("floor(%lf) = %lf \n\n", x2, floor(x2));
    printf("fmod(%lf, %lf) = %lf \n", x1, y = 2.4, fmod(x1, y));
    
    z = modf(x1, &y);
    printf("z = modf(%lf, &y) --> %lf(z) + %lf(y) \n", x1, z, y);
    return 0;
}

In [ ]:
// 02_math/px13_3.c (Practice 13.3: Pythagoras and pow)
#include <stdio.h>
#include <math.h>

int main() {
    double a = 20.0;
    double diag, area;
    
    diag = sqrt(a * a + a * a);
    area = pow(diag, 2);
    
    printf("대각선의 길이 : %lf \n", diag);
    printf("넓이 : %lf \n", area);
    return 0;
}

## 13.4 문자열-수치 변환 함수 (`stdlib.h`)
텍스트(문자열) 입력을 숫자 형태(`int`, `long`, `double`)로 변경하여 계산식에 사용하기 위한 함수들입니다. 본 예제 코드는 `day2/03_string_character/` 폴더에 들어있습니다.

- `atoi(const char *str)`: **A**scii **t**o **i**nteger. 문자열을 `int` 정수로 변환. 변환 불가능한 문자를 만나면 중단하고 이전까지 변환된 정수 반환 (첫 자부터 불가능하면 `0` 반환).
- `atol(const char *str)`: **A**scii **t**o **l**ong. 문자열을 `long` 정수로 변환.
- `atof(const char *str)`: **A**scii **t**o **f**loat. 문자열을 `double`형 실수로 변환.

In [ ]:
// 03_string_character/ex13_10.c
#include <stdio.h>
#include <stdlib.h>

int main() {
    int n1;
    long int n2;
    double f1;
    char *str1 = "1234.56";
    
    n1 = atoi(str1); // "."을 만났을 때 변환이 종료되므로 1234 반환
    printf("string = %s, integer = %d\n", str1, n1);
    n2 = atol(str1);
    printf("string = %s, long integer = %ld\n", str1, n2);
    f1 = atof(str1); // 실수 전체 1234.56 반환
    printf("string = %s, double = %lf\n", str1, f1);
    return 0;
}

## 13.5 & 13.6 문자 평가 및 변환 함수 (`ctype.h`)
개별 문자(`char`)의 상태를 판별하거나 형태를 일시적으로 변경해주는 함수들입니다. 본 코드들도 `day2/03_string_character/` 폴더에 분류되어 있습니다.

### 1. 문자 평가 함수 (참이면 0이 아닌 값, 거짓이면 0 반환)
- `isalnum(c)`: 알파벳 또는 숫자(A~Z, a~z, 0~9)이면 참.
- `isalpha(c)`: 알파벳(A~Z, a~z)이면 참.
- `isdigit(c)`: 10진수 숫자(0~9)이면 참.
- `isxdigit(c)`: 16진수 숫자(0~9, A~F, a~f)이면 참.
- `isspace(c)`: 공백 문자(공백, 탭, 개행 등)이면 참.
- `isupper(c)`: 대문자(A~Z)이면 참.
- `islower(c)`: 소문자(a~z)이면 참.
- `isprint(c)`: 출력 가능한 인쇄 문자이면 참.

### 2. 문자 변환 함수
- `tolower(c)`: 대문자를 소문자로 변환하여 반환 (대문자가 아닌 경우 변환 없이 `c` 반환).
- `toupper(c)`: 소문자를 대문자로 변환하여 반환.
- `toascii(c)`: 아스키 형식으로 변환 (하위 7비트만 취하고 나머지는 0으로 대체).

In [ ]:
// 03_string_character/ex13_11.c
#include <stdio.h>
#include <ctype.h>

int main() {
    char c = 'A';
    if (isalpha(c)) {
        if (isupper(c))
            printf("%c is alphabetic and upper case.\n", c);
        else
            printf("%c is alphabetic and lower case.\n", c);
    } else {
        printf("%c is not alphabetic.\n", c);
    }
    
    c = '5';
    if (isdigit(c)) {
        printf("%c is digit.\n", c);
    } else {
        printf("%c is not digit.\n", c);
    }
    return 0;
}

In [ ]:
// 03_string_character/ex13_12.c
#include <stdio.h>
#include <ctype.h>

int main() {
    int ascii, cap, low;
    char ch;
    
    printf("문자를 입력하세요 : ");
#ifdef _MSC_VER
    scanf_s("%c", &ch, 1);
#else
    scanf("%c", &ch);
#endif

    ascii = toascii(ch);
    printf("ascii = %d \n", ascii);
    cap = toupper(ch);
    printf("capital = %c \n", cap);
    low = tolower(ch);
    printf("lower = %c \n", low);
    return 0;
}